# Q&A Generation

Evaluates the RAG API's answers (`POST /v1/chat/completions`) against a small set of known question/answer pairs, by calling the API directly over HTTP — no Open WebUI involved.

**Prerequisites before running this notebook:**
- The API is running: `poetry run python src/api.py` (or `_start_api.sh`).
- Ollama is running locally, with the model(s) you want to test already pulled (`ollama pull <model>`).
- The Chroma index under `data/chroma/` has been built (see `src/rag_pipeline.py` / `notebooks/rag_pipeline_colab.ipynb`).

The API now honors the `model` field of each request (falling back to `Config.llm_model` only when it's empty or the synthetic `cat-gpt-rag` id Open WebUI sends by default) — so you can compare different locally-available Ollama models just by changing `MODEL_NAME` below and re-running, with no server restart needed.

In [1]:
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

In [2]:
BASE_URL = "http://127.0.0.1:8080"
RESULTS_DIR = Path("analysis")

## Connectivity check

In [3]:
try:
    response = requests.get(f"{BASE_URL}/v1/models", timeout=5)
    response.raise_for_status()
    print("API is reachable. Advertised models:", response.json())
except requests.exceptions.RequestException as exc:
    raise RuntimeError(
        f"Could not reach the API at {BASE_URL}. Is it running? "
        "Start it with `poetry run python src/api.py` and re-run this cell."
    ) from exc

API is reachable. Advertised models: {'object': 'list', 'data': [{'id': 'cat-gpt-rag', 'object': 'model', 'created': 1787004588, 'owned_by': 'local'}]}


## Evaluation dataset

A plain list of question/expected-answer pairs, edited directly here. Pre-populated from `Q&A.md` as a starting point — feel free to add, remove, or edit entries.

In [4]:
with open("./questions.txt", "r") as f:

    EVAL_QUESTIONS = [file.replace("\n", "") for file in f.readlines()]

EVAL_QUESTIONS

['What is chronic kidney disease in cats?',
 'How common is chronic kidney disease in cats over 15 years old?',
 'What are the early signs of feline diabetes?',
 'Can diabetes in cats be managed with diet alone?',
 'What is FLUTD?',
 'What causes feline lower urinary tract disease?',
 'What are the symptoms of hyperthyroidism in cats?',
 'How hyperthyroidism in cats are typically treated?',
 'Why is dental disease common in older cats?',
 'What are the warning signs of dental disease in cats?',
 'Why do vets discourage anesthesia-free dental cleanings?',
 'What is a normal body temperature for a cat?',
 'How often should a healthy adult cat visit the vet?',
 'Which vaccines are considered core for all cats?',
 "Which parts of a cat's body should vaccine injections avoid?",
 'What does FVRCP stand for?',
 'Are indoor cats still at risk of parasites?',
 'How can you tell if a cat has intestinal worms?',
 'What cause obesity in cats?',
 'How can you check if your cat is overweight?',
 'Wh

In [5]:
MODELS = [                   
    "gemma3:1b",       #815 MB
    "llama3.2:3b",     #2.0 GB
    "gemma3:4b",       #3.3 GB
    "llama3.2:1b"       #4.7 GB
]

# Concurrent requests per model. Models are still processed one at a time (loading
# several different models into Ollama simultaneously risks exceeding memory), but
# the 100 questions within a single model's batch are sent concurrently. Real
# throughput is capped by Ollama's own generation concurrency, not this number -
# start modest and tune based on observed wall-clock time.
MAX_WORKERS = 4

## Query the API

In [6]:
_SOURCES_MARKER_RE = re.compile(r"## Sources|No supporting sources were retrieved\.")


def _split_answer_from_sources(raw_content: str) -> tuple[str, str]:
    """The API concatenates the LLM answer with a code-generated sources block
    (twice, in fact - see src/rag/pipeline.py's _render_answer and src/api.py's
    chat_completions). Split on that block's fixed marker so evaluation only
    scores the model's own prose, keeping the sources text as a separate field.
    """
    match = _SOURCES_MARKER_RE.search(raw_content)
    if not match:
        return raw_content.strip(), ""
    return raw_content[: match.start()].strip(), raw_content[match.start():].strip()


def ask(question: str, model: str) -> dict:
    """POSTs a single question to /v1/chat/completions and returns the answer text + latency."""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": question}],
        "stream": False,
    }
    start = time.perf_counter()
    response = requests.post(f"{BASE_URL}/v1/chat/completions", json=payload, timeout=120)
    latency = time.perf_counter() - start
    response.raise_for_status()
    raw_content = response.json()["choices"][0]["message"]["content"]
    model_answer, sources_text = _split_answer_from_sources(raw_content)
    return {"answer": model_answer, "sources": sources_text, "latency_seconds": latency}

## Run the evaluation

In [7]:
from concurrent.futures import ThreadPoolExecutor


def _ask_row(index: int, question: str, model_name: str) -> dict:
    print(f"Asking question {index+1} for model {model_name}.")
    try:
        result = ask(question, model=model_name)
        answer = result["answer"]
        latency = result["latency_seconds"]
    except Exception as exc:
        answer = f"ERROR: {exc}"
        latency = float("nan")
    return {
        "_id": index+1,
        "question": question,
        "model_name": model_name,
        "model_answer": answer,
        # "sources": result["sources"],
        "latency_seconds": round(latency, 2),
        # "keyword_overlap_score": round(
        #     keyword_overlap_score(item["expected_answer"], result["answer"]), 2
        # ),
        # "manual_score": None,
    }


rows = []

# for model_name in MODELS:

model_name = MODELS[3]

print(f"The model {model_name} will answer the questions.")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(_ask_row, index, question, model_name)
        for index, question in enumerate(EVAL_QUESTIONS)
    ]
    rows.extend(future.result() for future in futures)


The model llama3.2:1b will answer the questions.
Asking question 1 for model llama3.2:1b.
Asking question 2 for model llama3.2:1b.
Asking question 3 for model llama3.2:1b.
Asking question 4 for model llama3.2:1b.
Asking question 5 for model llama3.2:1b.
Asking question 6 for model llama3.2:1b.
Asking question 7 for model llama3.2:1b.
Asking question 8 for model llama3.2:1b.
Asking question 9 for model llama3.2:1b.
Asking question 10 for model llama3.2:1b.
Asking question 11 for model llama3.2:1b.
Asking question 12 for model llama3.2:1b.
Asking question 13 for model llama3.2:1b.
Asking question 14 for model llama3.2:1b.
Asking question 15 for model llama3.2:1b.
Asking question 16 for model llama3.2:1b.
Asking question 17 for model llama3.2:1b.
Asking question 18 for model llama3.2:1b.
Asking question 19 for model llama3.2:1b.
Asking question 20 for model llama3.2:1b.
Asking question 21 for model llama3.2:1b.
Asking question 22 for model llama3.2:1b.
Asking question 23 for model llama3.

In [8]:
results_df = pd.DataFrame(rows) #.sort_values("keyword_overlap_score").reset_index(drop=True)

# Model tags like "gemma3:1b" contain a colon, which Windows/NTFS treats as an
# alternate-data-stream separator in filenames - writing to "gemma3:1b_results.json"
# silently creates an empty file named "gemma3" with the real content hidden in a
# stream, instead of erroring. Sanitize the name before building the path.
safe_model_name = re.sub(r"[^A-Za-z0-9._-]+", "-", model_name).strip("-")
results_df.to_json(f"./results/{safe_model_name}_results.json", indent=4, index=False, orient="records")

## Results

In [11]:
# pd.set_option("display.max_colwidth", None)
results_df

,_id,question,model_name,model_answer,latency_seconds
0,1,What is chronic kidney disease in cats?,gemma3:1b,Chronic kidney disease (CKD) is a progressive disease affecting up to 40% of cats over the age of 10 and 80% of cats over the age of 15. It’s a common condition in older cats and refers to the persistent loss of kidney function over time.,14.51
1,2,How common is chronic kidney disease in cats over 15 years old?,gemma3:1b,"Chronic kidney disease is one of the most prevalent diseases in older cats, affecting up to 40% of cats over the age of 10 and 80% of cats over the age of 15.",15.54
2,1,What is chronic kidney disease in cats?,llama3.2:1b,"Chronic Kidney Disease (CKD) in cats is a progressive disease that affects the kidneys, causing them to lose their ability to filter waste and excess fluids from the blood. It is a common problem in older cats, with up to 40% of cats over 10 years old and 80% of cats over 15 years old affected. CKD can lead to a buildup of toxic waste products in the bloodstream, dehydration, and a range of health problems, including hypertension, anemia, and loss of appetite. The disease is typically diagnosed in cats with a stable, well-hydrated state, and the International Renal Interest Society (IRIS) has developed a staging system to categorize cats with CKD based on blood and urine tests.",10.86
3,2,How common is chronic kidney disease in cats over 15 years old?,llama3.2:1b,"According to the sources, chronic kidney disease (CKD) is a common problem in older cats, affecting up to 80% of cats over the age of 15.",11.88


In [12]:
answers_df = results_df.pivot_table(
    index=["_id", "question"],
    columns="model_name",
    values=["model_answer"], #["model_answer", "latency_seconds"],
    aggfunc="first"
)

answers_df.columns = [model for field, model in answers_df.columns]
answers_df = answers_df.reset_index()

latency_df = results_df.pivot_table(
    index=["_id", "question"],
    columns="model_name",
    values=["latency_seconds"], #["model_answer", "latency_seconds"],
    aggfunc="first"
)

latency_df.columns = [model for field, model in latency_df.columns]
latency_df = latency_df.reset_index()

In [13]:
latency_df

,_id,question,gemma3:1b,llama3.2:1b
0,1,What is chronic kidney disease in cats?,14.51,10.86
1,2,How common is chronic kidney disease in cats over 15 years old?,15.54,11.88


In [14]:
answers_df

,_id,question,gemma3:1b,llama3.2:1b
0,1,What is chronic kidney disease in cats?,Chronic kidney disease (CKD) is a progressive disease affecting up to 40% of cats over the age of 10 and 80% of cats over the age of 15. It’s a common condition in older cats and refers to the persistent loss of kidney function over time.,"Chronic Kidney Disease (CKD) in cats is a progressive disease that affects the kidneys, causing them to lose their ability to filter waste and excess fluids from the blood. It is a common problem in older cats, with up to 40% of cats over 10 years old and 80% of cats over 15 years old affected. CKD can lead to a buildup of toxic waste products in the bloodstream, dehydration, and a range of health problems, including hypertension, anemia, and loss of appetite. The disease is typically diagnosed in cats with a stable, well-hydrated state, and the International Renal Interest Society (IRIS) has developed a staging system to categorize cats with CKD based on blood and urine tests."
1,2,How common is chronic kidney disease in cats over 15 years old?,"Chronic kidney disease is one of the most prevalent diseases in older cats, affecting up to 40% of cats over the age of 10 and 80% of cats over the age of 15.","According to the sources, chronic kidney disease (CKD) is a common problem in older cats, affecting up to 80% of cats over the age of 15."


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
safe_model_name = re.sub(r"[^A-Za-z0-9._-]+", "-", MODEL_NAME).strip("-")
output_path = RESULTS_DIR / f"results_{safe_model_name}_{timestamp}.csv"
results_df.to_csv(output_path, index=False)
print(f"Saved results to {output_path}")

## Comparing other Ollama models

To evaluate a different model:

1. `ollama pull <model-name>` if it isn't already local.
2. Set `MODEL_NAME = "<model-name>"` in the config cell above.
3. Re-run the notebook from the query cell down (no need to restart the API — `model` is now passed through per request).

Each run is saved to its own timestamped CSV under `analysis/`, tagged by model name, so you can load multiple result files afterward (e.g. with `pd.concat`) to compare models side by side.